In [ ]:
# ── Cell 0: Install Dependencies ──────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "torch", "torchaudio",
    "numpy", "pandas", "scikit-learn",
    "librosa", "soundfile",
    "tqdm", "matplotlib", "seaborn",
)

## Imports and Global Utilities

In [ ]:
import os, sys, random, json, time, warnings, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import soundfile as sf
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    balanced_accuracy_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc as sk_auc
)
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ── General utilities ─────────────────────────────────────────────────────────
def to_numpy(t):
    return t.detach().cpu().numpy() if isinstance(t, torch.Tensor) else np.array(t)

def flatten_dict(d, parent_key="", sep="/"):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

## Logging

In [ ]:
import logging
from datetime import datetime

LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = LOG_DIR / f"run_{run_id}.logger.info"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("mega_ia")

# ── Experiment result accumulator ─────────────────────────────────────────────
experiment_log = []

def log_result(exp_name: str, metrics: dict):
    entry = {"experiment": exp_name, "timestamp": run_id, **flatten_dict(metrics)}
    experiment_log.append(entry)
    logger.info(f"[{exp_name}] {metrics}")

def save_experiment_log(path: str = None):
    out = Path(path) if path else LOG_DIR / f"results_{run_id}.json"
    with open(out, "w") as f:
        json.dump(experiment_log, f, indent=2, default=str)
    logger.info(f"Experiment logger.info saved → {out}")

logger.info(f"Logging initialised | run_id={run_id} | logger.info={log_file}")

## Paths and Configurattions

In [ ]:
from pathlib import Path

VALPATH          = Path("datasets/validation_dataset/")
VAL_REAL_DIR     = VALPATH / "real"
VAL_FAKE_DIR     = VALPATH / "fake"
VAL_META_CSV     = VALPATH / "meta.csv"

LA_TEST_META        = Path("datasets/la_test/meta.csv")
IN_THE_WILD_META    = Path("datasets/in_the_wild_test/meta.csv")
DEEPFAKE_EVALS_META = Path("datasets/Deepfake-Evals-2024/audio-metadata-publish.csv")

CKPT_DIR = Path("checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = {
    "baseline":   CKPT_DIR / "rawnet2_baseline.pth",
    "mega_ia_v1": CKPT_DIR / "rawnet2_mega_ia_v1.pth",
    "mega_ia_v2": CKPT_DIR / "rawnet2_mega_ia_v2.pth",
    "la_parent":  Path("DeepSecure-Forensics/RawNet2 Parents/parent1_la/models/best_model.pth"),
    "itw_parent": Path("DeepSecure-Forensics/RawNet2 Parents/parent2_itw/models/best_auc.pth"),
}

SAMPLE_RATE  = 16000
MAX_DURATION = 4
MAX_SAMPLES  = SAMPLE_RATE * MAX_DURATION
BATCH_SIZE   = 32
NUM_WORKERS  = 4
NUM_EPOCHS   = 50
LR           = 1e-4
SEED         = 42

for p in [VALPATH, VAL_REAL_DIR, VAL_FAKE_DIR]:
    if not p.exists(): logger.warning(f'Path not found: {p}')
    else: logger.info(f'OK: {p}')

In [ ]:
# ── Cell 4: Device Setup ──────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True          # faster conv for fixed input size
    logger.info(f"CUDA device: {torch.cuda.get_device_name(0)} "
                f"| VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    logger.info("Device: Apple MPS")
else:
    DEVICE = torch.device("cpu")
    logger.info("Device: CPU")

logger.info(f"Active device → {DEVICE}")

## RawNet2 Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import numpy as np
import math
from torch.utils import data
from collections import OrderedDict
from torch.nn.parameter import Parameter
from torch.autograd import Variable
import pickle
import random


class SincConv(nn.Module):
    @staticmethod
    def to_mel(hz):
        return 2595 * np.log10(1 + hz / 700)

    @staticmethod
    def to_hz(mel):
        return 700 * (10 ** (mel / 2595) - 1)


    def __init__(self, device,out_channels, kernel_size,in_channels=1,sample_rate=16000,
                 stride=1, padding=0, dilation=1, bias=False, groups=1,freq_scale='Mel'):

        super(SincConv,self).__init__()


        if in_channels != 1:
            
            msg = "SincConv only support one input channel (here, in_channels = {%i})" % (in_channels)
            raise ValueError(msg)
        
        self.out_channels = out_channels+1
        self.kernel_size = kernel_size
        self.sample_rate=sample_rate

        # Forcing the filters to be odd (i.e, perfectly symmetrics)
        if kernel_size%2==0:
            self.kernel_size=self.kernel_size+1

        self.device=device   
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        
        if bias:
            raise ValueError('SincConv does not support bias.')
        if groups > 1:
            raise ValueError('SincConv does not support groups.')
        
        
        # initialize filterbanks using Mel scale
        NFFT = 512
        f=int(self.sample_rate/2)*np.linspace(0,1,int(NFFT/2)+1)


        if freq_scale == 'Mel':
            fmel=self.to_mel(f) # Hz to mel conversion
            fmelmax=np.max(fmel)
            fmelmin=np.min(fmel)
            filbandwidthsmel=np.linspace(fmelmin,fmelmax,self.out_channels+2)
            filbandwidthsf=self.to_hz(filbandwidthsmel) # Mel to Hz conversion
            self.freq=filbandwidthsf[:self.out_channels]

        elif freq_scale == 'Inverse-mel':
            fmel=self.to_mel(f) # Hz to mel conversion
            fmelmax=np.max(fmel)
            fmelmin=np.min(fmel)
            filbandwidthsmel=np.linspace(fmelmin,fmelmax,self.out_channels+2)
            filbandwidthsf=self.to_hz(filbandwidthsmel) # Mel to Hz conversion
            self.mel=filbandwidthsf[:self.out_channels]
            self.freq=np.abs(np.flip(self.mel)-1) ## invert mel scale

        
        else:
            fmelmax=np.max(f)
            fmelmin=np.min(f)
            filbandwidthsmel=np.linspace(fmelmin,fmelmax,self.out_channels+2)
            self.freq=filbandwidthsmel[:self.out_channels]
        
        self.hsupp=torch.arange(-(self.kernel_size-1)/2, (self.kernel_size-1)/2+1)
        self.band_pass=torch.zeros(self.out_channels-1,self.kernel_size)
    
       
        
    def forward(self,x):
        for i in range(len(self.freq)-1):
            fmin=self.freq[i]
            fmax=self.freq[i+1]
            hHigh=(2*fmax/self.sample_rate)*np.sinc(2*fmax*self.hsupp/self.sample_rate)
            hLow=(2*fmin/self.sample_rate)*np.sinc(2*fmin*self.hsupp/self.sample_rate)
            hideal=hHigh-hLow
            
            self.band_pass[i,:]=Tensor(np.hamming(self.kernel_size))*Tensor(hideal)
        
        band_pass_filter=self.band_pass.to(self.device)

        self.filters = (band_pass_filter).view(self.out_channels-1, 1, self.kernel_size)
        
        return F.conv1d(x, self.filters, stride=self.stride,
                        padding=self.padding, dilation=self.dilation,
                         bias=None, groups=1)


        
class Residual_block(nn.Module):
    def __init__(self, nb_filts, first = False):
        super(Residual_block, self).__init__()
        self.first = first
        
        if not self.first:
            self.bn1 = nn.BatchNorm1d(num_features = nb_filts[0])
        
        self.lrelu = nn.LeakyReLU(negative_slope=0.3)
        
        self.conv1 = nn.Conv1d(in_channels = nb_filts[0],
			out_channels = nb_filts[1],
			kernel_size = 3,
			padding = 1,
			stride = 1)
        
        self.bn2 = nn.BatchNorm1d(num_features = nb_filts[1])
        self.conv2 = nn.Conv1d(in_channels = nb_filts[1],
			out_channels = nb_filts[1],
			padding = 1,
			kernel_size = 3,
			stride = 1)
        
        if nb_filts[0] != nb_filts[1]:
            self.downsample = True
            self.conv_downsample = nn.Conv1d(in_channels = nb_filts[0],
				out_channels = nb_filts[1],
				padding = 0,
				kernel_size = 1,
				stride = 1)
            
        else:
            self.downsample = False
        self.mp = nn.MaxPool1d(3)
        
    def forward(self, x):
        identity = x
        if not self.first:
            out = self.bn1(x)
            out = self.lrelu(out)
        else:
            out = x
            
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.lrelu(out)
        out = self.conv2(out)
        
        if self.downsample:
            identity = self.conv_downsample(identity)
            
        out += identity
        out = self.mp(out)
        return out





class RawNet(nn.Module):
    def __init__(self, d_args, device):
        super(RawNet, self).__init__()

        
        self.device=device

        self.Sinc_conv=SincConv(device=self.device,
			out_channels = d_args['filts'][0],
			kernel_size = d_args['first_conv'],
                        in_channels = d_args['in_channels'],freq_scale='Mel'
        )
        
        self.first_bn = nn.BatchNorm1d(num_features = d_args['filts'][0])
        self.selu = nn.SELU(inplace=True)
        self.block0 = nn.Sequential(Residual_block(nb_filts = d_args['filts'][1], first = True))
        self.block1 = nn.Sequential(Residual_block(nb_filts = d_args['filts'][1]))
        self.block2 = nn.Sequential(Residual_block(nb_filts = d_args['filts'][2]))
        d_args['filts'][2][0] = d_args['filts'][2][1]
        self.block3 = nn.Sequential(Residual_block(nb_filts = d_args['filts'][2]))
        self.block4 = nn.Sequential(Residual_block(nb_filts = d_args['filts'][2]))
        self.block5 = nn.Sequential(Residual_block(nb_filts = d_args['filts'][2]))
        self.avgpool = nn.AdaptiveAvgPool1d(1)

        self.fc_attention0 = self._make_attention_fc(in_features = d_args['filts'][1][-1],
            l_out_features = d_args['filts'][1][-1])
        self.fc_attention1 = self._make_attention_fc(in_features = d_args['filts'][1][-1],
            l_out_features = d_args['filts'][1][-1])
        self.fc_attention2 = self._make_attention_fc(in_features = d_args['filts'][2][-1],
            l_out_features = d_args['filts'][2][-1])
        self.fc_attention3 = self._make_attention_fc(in_features = d_args['filts'][2][-1],
            l_out_features = d_args['filts'][2][-1])
        self.fc_attention4 = self._make_attention_fc(in_features = d_args['filts'][2][-1],
            l_out_features = d_args['filts'][2][-1])
        self.fc_attention5 = self._make_attention_fc(in_features = d_args['filts'][2][-1],
            l_out_features = d_args['filts'][2][-1])

        self.bn_before_gru = nn.BatchNorm1d(num_features = d_args['filts'][2][-1])
        self.gru = nn.GRU(input_size = d_args['filts'][2][-1],
			hidden_size = d_args['gru_node'],
			num_layers = d_args['nb_gru_layer'],
			batch_first = True)

        
        self.fc1_gru = nn.Linear(in_features = d_args['gru_node'],
			out_features = d_args['nb_fc_node'])
       
        self.fc2_gru = nn.Linear(in_features = d_args['nb_fc_node'],
			out_features = d_args['nb_classes'],bias=True)
			
       
        self.sig = nn.Sigmoid()
        
        
    def forward(self, x, y = None,is_test=False):
        
        
        nb_samp = x.shape[0]
        len_seq = x.shape[1]
        x=x.view(nb_samp,1,len_seq)
        
        x = self.Sinc_conv(x)    # Fixed sinc filters convolution
        x = F.max_pool1d(torch.abs(x), 3)
        x = self.first_bn(x)
        x = self.selu(x)
        
        x0 = self.block0(x)
        y0 = self.avgpool(x0).view(x0.size(0), -1) # torch.Size([batch, filter])
        y0 = self.fc_attention0(y0)
        y0 = self.sig(y0).view(y0.size(0), y0.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x0 * y0 + y0  # (batch, filter, time) x (batch, filter, 1)
        

        x1 = self.block1(x)
        y1 = self.avgpool(x1).view(x1.size(0), -1) # torch.Size([batch, filter])
        y1 = self.fc_attention1(y1)
        y1 = self.sig(y1).view(y1.size(0), y1.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x1 * y1 + y1 # (batch, filter, time) x (batch, filter, 1)

        x2 = self.block2(x)
        y2 = self.avgpool(x2).view(x2.size(0), -1) # torch.Size([batch, filter])
        y2 = self.fc_attention2(y2)
        y2 = self.sig(y2).view(y2.size(0), y2.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x2 * y2 + y2 # (batch, filter, time) x (batch, filter, 1)

        x3 = self.block3(x)
        y3 = self.avgpool(x3).view(x3.size(0), -1) # torch.Size([batch, filter])
        y3 = self.fc_attention3(y3)
        y3 = self.sig(y3).view(y3.size(0), y3.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x3 * y3 + y3 # (batch, filter, time) x (batch, filter, 1)

        x4 = self.block4(x)
        y4 = self.avgpool(x4).view(x4.size(0), -1) # torch.Size([batch, filter])
        y4 = self.fc_attention4(y4)
        y4 = self.sig(y4).view(y4.size(0), y4.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x4 * y4 + y4 # (batch, filter, time) x (batch, filter, 1)

        x5 = self.block5(x)
        y5 = self.avgpool(x5).view(x5.size(0), -1) # torch.Size([batch, filter])
        y5 = self.fc_attention5(y5)
        y5 = self.sig(y5).view(y5.size(0), y5.size(1), -1)  # torch.Size([batch, filter, 1])
        x = x5 * y5 + y5 # (batch, filter, time) x (batch, filter, 1)

        x = self.bn_before_gru(x)
        x = self.selu(x)
        x = x.permute(0, 2, 1)     #(batch, filt, time) >> (batch, time, filt)
        self.gru.flatten_parameters()
        x, _ = self.gru(x)
        x = x[:,-1,:]
        x = self.fc1_gru(x)
        x = self.fc2_gru(x)

        if not is_test:
            output = x
            return output

        else:
            output=F.softmax(x,dim=1)
            return output
      
       
        

    def _make_attention_fc(self, in_features, l_out_features):

        l_fc = []
        
        l_fc.append(nn.Linear(in_features = in_features,
			        out_features = l_out_features))

        

        return nn.Sequential(*l_fc)


    def _make_layer(self, nb_blocks, nb_filts, first = False):
        layers = []
        #def __init__(self, nb_filts, first = False):
        for i in range(nb_blocks):
            first = first if i == 0 else False
            layers.append(Residual_block(nb_filts = nb_filts,
				first = first))
            if i == 0: nb_filts[0] = nb_filts[1]
            
        return nn.Sequential(*layers)

    def summary(self, input_size, batch_size=-1, device="cuda", print_fn = None):
        if print_fn == None: printfn = print
        model = self
        
        def register_hook(module):
            def hook(module, input, output):
                class_name = str(module.__class__).split(".")[-1].split("'")[0]
                module_idx = len(summary)
                
                m_key = "%s-%i" % (class_name, module_idx + 1)
                summary[m_key] = OrderedDict()
                summary[m_key]["input_shape"] = list(input[0].size())
                summary[m_key]["input_shape"][0] = batch_size
                if isinstance(output, (list, tuple)):
                    summary[m_key]["output_shape"] = [
						[-1] + list(o.size())[1:] for o in output
					]
                else:
                    summary[m_key]["output_shape"] = list(output.size())
                    if len(summary[m_key]["output_shape"]) != 0:
                        summary[m_key]["output_shape"][0] = batch_size
                        
                params = 0
                if hasattr(module, "weight") and hasattr(module.weight, "size"):
                    params += torch.prod(torch.LongTensor(list(module.weight.size())))
                    summary[m_key]["trainable"] = module.weight.requires_grad
                if hasattr(module, "bias") and hasattr(module.bias, "size"):
                    params += torch.prod(torch.LongTensor(list(module.bias.size())))
                summary[m_key]["nb_params"] = params
                
            if (
				not isinstance(module, nn.Sequential)
				and not isinstance(module, nn.ModuleList)
				and not (module == model)
			):
                hooks.append(module.register_forward_hook(hook))
                
        device = device.lower()
        assert device in [
			"cuda",
			"cpu",
		], "Input device is not valid, please specify 'cuda' or 'cpu'"
        
        if device == "cuda" and torch.cuda.is_available():
            dtype = torch.cuda.FloatTensor
        else:
            dtype = torch.FloatTensor
        if isinstance(input_size, tuple):
            input_size = [input_size]
        x = [torch.rand(2, *in_size).type(dtype) for in_size in input_size]
        summary = OrderedDict()
        hooks = []
        model.apply(register_hook)
        model(*x)
        for h in hooks:
            h.remove()
            
        print_fn("----------------------------------------------------------------")
        line_new = "{:>20}  {:>25} {:>15}".format("Layer (type)", "Output Shape", "Param #")
        print_fn(line_new)
        print_fn("================================================================")
        total_params = 0
        total_output = 0
        trainable_params = 0
        for layer in summary:
            # input_shape, output_shape, trainable, nb_params
            line_new = "{:>20}  {:>25} {:>15}".format(
				layer,
				str(summary[layer]["output_shape"]),
				"{0:,}".format(summary[layer]["nb_params"]),
			)
            total_params += summary[layer]["nb_params"]
            total_output += np.prod(summary[layer]["output_shape"])
            if "trainable" in summary[layer]:
                if summary[layer]["trainable"] == True:
                    trainable_params += summary[layer]["nb_params"]
            print_fn(line_new)


## Dataset Classes

In [ ]:
# ── Dataset Classes ────────────────────────────────────────────────────────────
import warnings
from torch.utils.data import Dataset

def pad_random(x: np.ndarray, max_len: int = 64600) -> np.ndarray:
    x_len = x.shape[0]
    if x_len > max_len:
        stt = np.random.randint(x_len - max_len)
        return x[stt:stt + max_len]
    return np.tile(x, (int(max_len / x_len) + 1))[:max_len]


class ValidationDataset(Dataset):
    """Local validation dataset — real/ and fake/ dirs with meta.csv.
    TODO: confirm meta.csv column names once dataset is finalised.
    Expected columns: sample_name, is_fake (1=fake, 0=real)
    """
    def __init__(self, root_dir: str):
        self.root_dir = root_dir
        df = pd.read_csv(os.path.join(root_dir, 'meta.csv'))
        self.df = df.reset_index(drop=True)
        # TODO: update column names below to match actual meta.csv schema
        self.n_fake = int(self.df['is_fake'].sum())
        self.n_real = len(self.df) - self.n_fake

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        subdir = 'fake' if row['is_fake'] == 1 else 'real'
        # TODO: update 'sample_name' to actual filename column if different
        path   = os.path.join(self.root_dir, subdir, row['sample_name'])
        x, _   = sf.read(path)
        if x.ndim > 1: x = x.mean(axis=1)
        label  = 0 if row['is_fake'] == 1 else 1   # 0=spoof, 1=bonafide
        return torch.FloatTensor(pad_random(x)), label


class LATestDataset(Dataset):
    """ASVspoof LA test set.
    CSV columns: Filename, Ground Truth
    Filename includes subdir prefix e.g. real/LA_E_xxx.flac or fake/LA_E_xxx.flac
    Ground Truth: 'Real' | 'Fake'
    """
    TARGET_SR = 16000

    def __init__(self, meta_csv: str):
        self.root_dir = os.path.dirname(meta_csv)
        df = pd.read_csv(meta_csv)
        self.df = df.reset_index(drop=True)
        self.n_real = int((self.df['Ground Truth'] == 'Real').sum())
        self.n_fake = int((self.df['Ground Truth'] == 'Fake').sum())

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        # Filename already contains subdir e.g. real/LA_E_xxx.flac
        path  = os.path.join(self.root_dir, str(row['Filename']))
        x, sr = sf.read(path)
        if x.ndim > 1: x = x.mean(axis=1)
        if sr != self.TARGET_SR:
            x = librosa.resample(x, orig_sr=sr, target_sr=self.TARGET_SR)
        label = 1 if row['Ground Truth'] == 'Real' else 0
        return torch.FloatTensor(pad_random(x)), label


class InTheWildDataset(Dataset):
    """In-the-Wild test set.
    CSV columns: file, speaker, label
    label: 'bona-fide' | 'spoof'
    """
    TARGET_SR = 16000

    def __init__(self, meta_csv: str):
        self.root_dir = os.path.dirname(meta_csv)
        df = pd.read_csv(meta_csv)
        self.df = df.reset_index(drop=True)
        self.n_real = int((self.df['label'] == 'bona-fide').sum())
        self.n_fake = int((self.df['label'] == 'spoof').sum())

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.root_dir, str(row['file']))
        x, sr = sf.read(path)
        if x.ndim > 1: x = x.mean(axis=1)
        if sr != self.TARGET_SR:
            x = librosa.resample(x, orig_sr=sr, target_sr=self.TARGET_SR)
        label = 1 if row['label'] == 'bona-fide' else 0
        return torch.FloatTensor(pad_random(x)), label


class DeepfakeEvalsDataset(Dataset):
    """Deepfake-Evals-2024.
    CSV columns: Filename, Date, Ground Truth, Public Comments, Finetuning Set
    Ground Truth: 'Real' | 'Fake'
    Supports three modes via `indices` param (used for splits — see split utility below).
    """
    TARGET_SR = 16000

    def __init__(self, root_dir: str, cache_dir: str = None, indices=None):
        self.root_dir = root_dir
        df = pd.read_csv(os.path.join(root_dir, 'audio-metadata-publish.csv'))
        df = df[df['Filename'].notna()].copy()
        df['Filename'] = df['Filename'].astype(str).str.strip()

        cache_path = os.path.join(cache_dir, 'deepfake_valid_indices.json') if cache_dir else None

        if cache_path and os.path.exists(cache_path):
            with open(cache_path) as f:
                cached = json.load(f)
            good = cached['good_indices']
            logger.info(f'Loaded cached file list ({len(good)} valid, {cached.get("skip_count",0)} skipped).')
        else:
            logger.info('Pre-flighting Deepfake-Evals files (runs once, then cached)...')
            good, skip = [], []
            total = len(df)
            for pos, (i, row) in enumerate(df.iterrows()):
                if pos % 100 == 0:
                    logger.info(f'  Pre-flight: {pos}/{total}...')
                path = os.path.join(root_dir, row['Filename'])
                if not os.path.isfile(path):
                    skip.append(row['Filename']); continue
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        librosa.load(path, sr=None, mono=True, duration=0.1)
                    good.append(i)
                except Exception:
                    skip.append(row['Filename'])
            if cache_path:
                with open(cache_path, 'w') as f:
                    json.dump({'good_indices': good, 'skip_count': len(skip)}, f)
                logger.info(f'Valid file list cached → {cache_path}')

        self.df = df.loc[good].reset_index(drop=True)

        # Apply external index mask (used for disjoint splits)
        if indices is not None:
            self.df = self.df.loc[indices].reset_index(drop=True)

        self.n_real = int((self.df['Ground Truth'] == 'Real').sum())
        self.n_fake = int((self.df['Ground Truth'] == 'Fake').sum())

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.root_dir, str(row['Filename']))
        x, _  = librosa.load(path, sr=self.TARGET_SR, mono=True)
        label = 1 if row['Ground Truth'] == 'Real' else 0
        return torch.FloatTensor(pad_random(x)), label


def split_deepfake_evals(root_dir: str, cache_dir: str,
                         val_frac: float = 0.15,
                         seed: int = SEED):
    """Return three disjoint DeepfakeEvalsDataset instances:
        full_ds   — 100% (all valid files)
        val_ds    — 15%  balanced: equal Real/Fake counts  (7.5% each)
        test_ds   — 85%  remainder (all files not in val)

    Constraint: val set is balanced — Real count == Fake count.
    The smaller of (target_real, target_fake) determines val size per class.
    """
    rng = np.random.default_rng(seed)

    # Build full dataset first to get the validated df
    full_ds = DeepfakeEvalsDataset(root_dir, cache_dir=cache_dir)
    df = full_ds.df.copy()

    real_idx = df.index[df['Ground Truth'] == 'Real'].tolist()
    fake_idx = df.index[df['Ground Truth'] == 'Fake'].tolist()

    n_total      = len(df)
    n_val_target = int(n_total * val_frac)          # 15% of total
    n_per_class  = n_val_target // 2                # equal split
    # Clamp to available counts
    n_per_class  = min(n_per_class, len(real_idx), len(fake_idx))

    rng.shuffle(real_idx := np.array(real_idx))
    rng.shuffle(fake_idx := np.array(fake_idx))

    val_real = real_idx[:n_per_class].tolist()
    val_fake = fake_idx[:n_per_class].tolist()
    val_idx  = val_real + val_fake

    val_idx_set  = set(val_idx)
    test_idx     = [i for i in df.index.tolist() if i not in val_idx_set]

    val_ds  = DeepfakeEvalsDataset(root_dir, cache_dir=cache_dir, indices=val_idx)
    test_ds = DeepfakeEvalsDataset(root_dir, cache_dir=cache_dir, indices=test_idx)

    logger.info(f'DFE full : {len(full_ds):,}  ({full_ds.n_real:,} real / {full_ds.n_fake:,} fake)')
    logger.info(f'DFE val  : {len(val_ds):,}  ({val_ds.n_real:,} real / {val_ds.n_fake:,} fake)  [{len(val_ds)/len(full_ds)*100:.1f}%]')
    logger.info(f'DFE test : {len(test_ds):,}  ({test_ds.n_real:,} real / {test_ds.n_fake:,} fake)  [{len(test_ds)/len(full_ds)*100:.1f}%]')
    assert len(val_idx_set & set(test_idx)) == 0, 'Disjoint constraint violated!'
    assert val_ds.n_real == val_ds.n_fake,         'Val balance constraint violated!'

    return full_ds, val_ds, test_ds


logger.info('Dataset classes defined.')

## Load Datasets (Full — No Subsampling)

In [ ]:
# ── Load Datasets ──────────────────────────────────────────────────────────────
from torch.utils.data import DataLoader

logger.info('Loading validation dataset...')
val_ds = ValidationDataset(str(VALPATH))
logger.info(f'Val:           {len(val_ds):,} samples  ({val_ds.n_real:,} real / {val_ds.n_fake:,} fake)')

logger.info('Loading LA test set...')
la_ds = LATestDataset(str(LA_TEST_META))
logger.info(f'LA test:       {len(la_ds):,} samples  ({la_ds.n_real:,} real / {la_ds.n_fake:,} fake)')

logger.info('Loading In-the-Wild test set...')
itw_ds = InTheWildDataset(str(IN_THE_WILD_META))
logger.info(f'In-the-Wild:   {len(itw_ds):,} samples  ({itw_ds.n_real:,} real / {itw_ds.n_fake:,} fake)')

logger.info('Loading Deepfake-Evals-2024 (full + splits)...')
dfe_root     = str(DEEPFAKE_EVALS_META.parent)
dfe_full_ds, dfe_val_ds, dfe_test_ds = split_deepfake_evals(
    root_dir=dfe_root, cache_dir=str(CKPT_DIR), val_frac=0.15, seed=SEED
)

_pin = (str(DEVICE) == 'cuda')
_nw  = NUM_WORKERS

val_loader      = DataLoader(val_ds,      batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)
la_loader       = DataLoader(la_ds,       batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)
itw_loader      = DataLoader(itw_ds,      batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)
dfe_full_loader = DataLoader(dfe_full_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)
dfe_val_loader  = DataLoader(dfe_val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)
dfe_test_loader = DataLoader(dfe_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=_pin)

logger.info(f'val_loader      : {len(val_loader):,} batches')
logger.info(f'la_loader       : {len(la_loader):,} batches')
logger.info(f'itw_loader      : {len(itw_loader):,} batches')
logger.info(f'dfe_full_loader : {len(dfe_full_loader):,} batches')
logger.info(f'dfe_val_loader  : {len(dfe_val_loader):,} batches')
logger.info(f'dfe_test_loader : {len(dfe_test_loader):,} batches')

# Smoke-test
logger.info('Smoke-testing loaders...')
for name, ldr in [('val', val_loader), ('la', la_loader), ('itw', itw_loader),
                  ('dfe_full', dfe_full_loader), ('dfe_val', dfe_val_loader), ('dfe_test', dfe_test_loader)]:
    xb, yb = next(iter(ldr))
    logger.info(f'{name:>10} batch: {tuple(xb.shape)}  labels={yb[:4].tolist()}')
    del xb, yb
logger.info('All loaders confirmed.')

## Load Parent Checkpoints

In [ ]:
d_args = {
    'filts'       : [20, [20, 20], [20, 128]],
    'first_conv'  : 1024,
    'in_channels' : 1,
    'gru_node'    : 1024,
    'nb_gru_layer': 3,
    'nb_fc_node'  : 1024,
    'nb_classes'  : 2,
}

logger.info('Loading parent checkpoints...')

def load_parent(path, label):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        sd = ckpt['model_state_dict']
        logger.info(f'{label}: full checkpoint, epoch={ckpt.get("epoch","?")}')
    else:
        sd = ckpt
        logger.info(f'{label}: legacy weights-only checkpoint')
    logger.info(f'{label}: {len(sd)} keys')
    return sd

p1_state = load_parent(MODEL_PATHS['la_parent'],  'LA parent')
p2_state = load_parent(MODEL_PATHS['itw_parent'], 'ITW parent')

# ── Key consistency checks ────────────────────────────────────────────────────
assert set(p1_state.keys()) == set(p2_state.keys()), \
    f'Parent key mismatch!\n  LA only : {set(p1_state.keys()) - set(p2_state.keys())}\n  ITW only: {set(p2_state.keys()) - set(p1_state.keys())}'

# Validate against live v2 model (class is RawNet, not RawNet2)
_d = {
    'filts'       : [20, [20, 20], [20, 128]],
    'first_conv'  : 1024,
    'in_channels' : 1,
    'gru_node'    : 1024,
    'nb_gru_layer': 3,
    'nb_fc_node'  : 1024,
    'nb_classes'  : 2,
}
_probe = RawNet(_d, DEVICE).to(DEVICE)
_model_keys = set(_probe.state_dict().keys())
del _probe

missing = _model_keys - set(p1_state.keys())
extra   = set(p1_state.keys()) - _model_keys

if missing or extra:
    if missing: logger.warning(f'Missing in checkpoint: {missing}')
    if extra:   logger.warning(f'Extra in checkpoint  : {extra}')
    raise RuntimeError('Parent keys do not match v2 RawNet. Inspect above.')

logger.info('Both parents loaded and keys verified against v2 RawNet.')

## Core Utilities: Feature Extraction, CKA, Metrics

In [ ]:
# ── Core Utilities: Feature Extraction, CKA, Metrics ──────────────────────────

def compute_linear_cka(x: torch.Tensor, y: torch.Tensor) -> float:
    """Linear Centred Kernel Alignment. Both tensors: (N, D) on CPU."""
    x = x - x.mean(0)
    y = y - y.mean(0)
    num   = torch.norm(torch.matmul(x.t(), y)) ** 2
    denom = (torch.norm(torch.matmul(x.t(), x)) *
             torch.norm(torch.matmul(y.t(), y)) + 1e-8)
    return (num / denom).item()


def extract_reference_features(model, loader, device):
    """
    Extract features at 3 depths from parent-1 for CKA reference.
    Returns dict: 'early' (block1), 'mid' (block3), 'final' (GRU last step).
    All tensors shape (N, D), CPU.
    """
    model.eval()
    fe, fm, ff = [], [], []

    def h_early(m, inp, out):
        fe.append(F.adaptive_avg_pool1d(out.detach().cpu(), 1).squeeze(-1))
    def h_mid(m, inp, out):
        fm.append(F.adaptive_avg_pool1d(out.detach().cpu(), 1).squeeze(-1))
    def h_gru(m, inp, out):
        seq, _ = out
        ff.append(seq[:, -1, :].detach().cpu())

    hooks = [
        model.block1.register_forward_hook(h_early),
        model.block3.register_forward_hook(h_mid),
        model.gru.register_forward_hook(h_gru),
    ]
    with torch.no_grad():
        for x, _ in tqdm(loader, desc='  Extracting P1 features', leave=False):
            model(x.to(device), is_test=True)
    for h in hooks: h.remove()

    return {
        'early': torch.cat(fe),
        'mid':   torch.cat(fm),
        'final': torch.cat(ff),
    }


def full_metrics(labels, preds, scores):
    """Compute all evaluation metrics given labels, hard preds, and soft scores."""
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    return {
        'accuracy'  : float(accuracy_score(labels, preds)),
        'auc'       : float(roc_auc_score(labels, scores)),
        'bal_acc'   : float(balanced_accuracy_score(labels, preds)),
        'precision' : float(precision_score(labels, preds, zero_division=0)),
        'recall'    : float(recall_score(labels, preds, zero_division=0)),
        'f1'        : float(f1_score(labels, preds, zero_division=0)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'confusion_matrix': cm.tolist(),
    }


def evaluate_candidate(model, loader, device, p1_feats, config):
    """
    Evaluate one candidate model on the validation set.
    Returns (metrics_dict, fitness_score).

    Fitness (MeGA-IA v2 formulation from architecture doc):
        S(c) = AUC - λ1·CKA_multi + λ2·Confidence

    CKA_multi: simple mean across [early, mid, final] layers.
    Confidence: mean(p² + (1-p)²) — confidence bonus (added, not subtracted).
    Entropy penalty (v1 style): mean(-Σ p·log2(p)) — kept for legacy configs.
    """
    model.eval()
    all_scores, all_labels = [], []
    cf_e, cf_m, cf_f = [], [], []

    def h_e(m, inp, out):
        cf_e.append(F.adaptive_avg_pool1d(out.detach().cpu(), 1).squeeze(-1))
    def h_m(m, inp, out):
        cf_m.append(F.adaptive_avg_pool1d(out.detach().cpu(), 1).squeeze(-1))
    def h_g(m, inp, out):
        seq, _ = out
        cf_f.append(seq[:, -1, :].detach().cpu())

    hooks = [
        model.block1.register_forward_hook(h_e),
        model.block3.register_forward_hook(h_m),
        model.gru.register_forward_hook(h_g),
    ]
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device), is_test=True)
            all_scores.append(out[:, 1].cpu().numpy())
            all_labels.append(y.numpy())
    for h in hooks: h.remove()

    scores = np.concatenate(all_scores)
    labels = np.concatenate(all_labels)
    preds  = (scores > 0.5).astype(int)

    feat_e = torch.cat(cf_e)
    feat_m = torch.cat(cf_m)
    feat_f = torch.cat(cf_f)

    met = full_metrics(labels, preds, scores)
    met['scores'] = scores.tolist()
    met['labels'] = labels.tolist()

    # ── Confidence bonus (v2 doc): mean(p² + (1-p)²) ─────────────────────
    p = scores.clip(1e-8, 1 - 1e-8)
    met['confidence'] = float(np.mean(p ** 2 + (1 - p) ** 2))

    # ── Legacy entropy penalty (v1 style, kept for E01-E10 compat) ────────
    p_mat = np.stack([1 - p, p], axis=1)
    met['entropy'] = float(-np.mean(np.sum(p_mat * np.log2(p_mat), axis=1)))

    # ── CKA: mode controlled by config ────────────────────────────────────
    cka_mode = config.get('cka_mode', 'final')
    if cka_mode == 'multilayer':
        cka_e = compute_linear_cka(feat_e, p1_feats['early'])
        cka_m = compute_linear_cka(feat_m, p1_feats['mid'])
        cka_f = compute_linear_cka(feat_f, p1_feats['final'])
        # v2 doc: simple mean across layers (equal weight)
        cka_score = (cka_e + cka_m + cka_f) / 3.0
        met['cka_early'] = float(cka_e)
        met['cka_mid']   = float(cka_m)
        met['cka_final'] = float(cka_f)
    else:
        cka_score = compute_linear_cka(feat_f, p1_feats['final'])
    met['cka'] = float(cka_score)

    # ── Composite fitness ─────────────────────────────────────────────────
    ft_type = config.get('fitness_type', 'accuracy')
    base    = {'accuracy': met['accuracy'],
               'auc':      met['auc'],
               'balanced': met['bal_acc']}[ft_type]

    penalty_mode = config.get('penalty_mode', 'entropy')   # 'entropy' | 'confidence'
    if penalty_mode == 'confidence':
        # v2 doc: AUC - λ1·CKA + λ2·Confidence  (confidence is a BONUS)
        fitness = base - config['lambda1'] * cka_score + config['lambda2'] * met['confidence']
    else:
        # v1 legacy: AUC - λ1·interference - λ2·entropy
        fitness = base - config['lambda1'] * cka_score - config['lambda2'] * met['entropy']

    met['fitness'] = float(fitness)
    return met, float(fitness)


# ── Checkpoint helpers ─────────────────────────────────────────────────────────
def save_exp_results(ckpt_dir, exp_id, best_val_metrics, history, best_genome):
    out = {
        'exp_id': exp_id,
        'best_val_metrics': {k: v for k, v in best_val_metrics.items()
                             if k not in ('scores', 'labels')},
        'history':     history,
        'best_genome': best_genome.tolist() if hasattr(best_genome, 'tolist') else best_genome,
    }
    path = os.path.join(ckpt_dir, f'{exp_id}_results.json')
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)
    logger.info(f'Results saved → {path}')


def load_exp_results(ckpt_dir, exp_id):
    path = os.path.join(ckpt_dir, f'{exp_id}_results.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def save_test_results(ckpt_dir, exp_id, dataset_key, test_metrics):
    """dataset_key: 'dfe_full' | 'la' | 'itw'"""
    out = {k: v for k, v in test_metrics.items() if k not in ('scores', 'labels')}
    path = os.path.join(ckpt_dir, f'{exp_id}_test_{dataset_key}.json')
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)


def load_test_results(ckpt_dir, exp_id, dataset_key):
    path = os.path.join(ckpt_dir, f'{exp_id}_test_{dataset_key}.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


logger.info('Core utilities defined.')

## MeGA-IA Evolutionary Engine

In [ ]:
# ── MeGA-IA Evolutionary Engine ────────────────────────────────────────────────

def run_mega_ia(config, p1_state, p2_state, d_args, device,
                val_loader, p1_feats, str(CKPT_DIR)):
    """
    Run one MeGA-IA experiment to completion with full checkpointing.

    alpha_mode:
      'global'   — single alpha scalar
      'segment3' — 3 independent alphas: early / mid / late layers (v2 default)

    crossover_mode (from config, default 'blend'):
      'blend'    — β·g1 + (1-β)·g2  (v2 doc formulation)
      'uniform'  — per-gene coin-flip  (v1 legacy)

    Returns (best_val_metrics, history, best_genome).
    """
    exp_id      = config['id']
    evo_ckpt    = os.path.join(str(CKPT_DIR), f'{exp_id}_evo_state.json')
    weights_pth = os.path.join(str(CKPT_DIR), f'{exp_id}_best_weights.pth')

    # ── Genome setup ──────────────────────────────────────────────────────
    float_keys  = [k for k in p1_state if p1_state[k].dtype.is_floating_point]
    n_fl        = len(float_keys)
    all_keys    = list(p1_state.keys())
    alpha_mode  = config.get('alpha_mode', 'global')
    genome_size = 1 if alpha_mode == 'global' else 3

    def make_model(genome):
        m  = RawNet(d_args, device).to(device)
        sd, fi = {}, 0
        for k in all_keys:
            v1 = p1_state[k].to(device)
            v2 = p2_state[k].to(device)
            if not v1.dtype.is_floating_point:
                sd[k] = v1.clone()
            else:
                if alpha_mode == 'global':
                    a = float(genome[0])
                else:  # segment3: early=0, mid=1, late=2
                    seg = min(fi * 3 // n_fl, 2)
                    a   = float(genome[seg])
                sd[k] = a * v1 + (1.0 - a) * v2
                fi += 1
        m.load_state_dict(sd)
        return m

    def rnd_genome():
        return np.random.uniform(0.0, 1.0, genome_size)

    def crossover(g1, g2):
        mode = config.get('crossover_mode', 'blend')
        if mode == 'blend':
            # v2 doc: β·g1 + (1-β)·g2 with single β
            beta = np.random.rand()
            return beta * g1 + (1.0 - beta) * g2
        else:
            # v1 legacy: uniform per-gene mask
            mask = np.random.rand(genome_size) < 0.5
            return np.where(mask, g1, g2)

    def mutate(genome, gen):
        decay = math.exp(-0.5 * gen / max(config['generations'], 1))
        sigma = config.get('sigma_base', 0.10) * decay
        return np.clip(genome + np.random.normal(0, sigma, genome_size), 0.0, 1.0)

    def tournament(pop, fits, k=4):
        idx    = np.random.choice(len(pop), min(k, len(pop)), replace=False)
        best_i = idx[np.argmax([fits[i] for i in idx])]
        return pop[best_i].copy()

    # ── Initialize / resume ───────────────────────────────────────────────
    start_gen    = 0
    population   = [rnd_genome() for _ in range(config['pop_size'])]
    best_genome  = population[0].copy()
    best_fitness = -np.inf
    best_val_met = None
    stagnation   = 0
    hist_keys    = ['best_fit', 'mean_fit', 'worst_fit', 'best_auc', 'best_bal',
                    'best_f1', 'mean_auc', 'diversity', 'best_prec', 'best_rec',
                    'best_cka', 'best_conf']
    history = {k: [] for k in hist_keys}

    if os.path.exists(evo_ckpt):
        try:
            with open(evo_ckpt) as f:
                ec = json.load(f)
            start_gen    = ec['last_gen'] + 1
            population   = [np.array(g) for g in ec['population']]
            best_genome  = np.array(ec['best_genome'])
            best_fitness = ec['best_fitness']
            best_val_met = ec.get('best_val_met')
            stagnation   = ec.get('stagnation', 0)
            history      = ec['history']
            logger.info(f'  ♻️  [{exp_id}] Resumed from gen {start_gen}')
        except Exception as e:
            logger.warning(f'  Evo checkpoint corrupt ({e}) — starting fresh')

    total_gens = config['generations']
    pop_sz     = config['pop_size']

    # ── Evolutionary loop ─────────────────────────────────────────────────
    for gen in range(start_gen, total_gens):
        t0  = time.time()
        logger.info(f'')
        logger.info(f'┌─ [{config["name"]}]  Gen {gen+1:2d}/{total_gens}')

        gen_fits, gen_mets = [], []

        for ci, genome in enumerate(population):
            model = make_model(genome)
            met, fit = evaluate_candidate(model, val_loader, device, p1_feats, config)
            gen_fits.append(fit)
            gen_mets.append(met)
            del model
            if str(device) == 'cuda': torch.cuda.empty_cache()

            cka_str = (f'cka_e={met["cka_early"]:.3f} cka_m={met["cka_mid"]:.3f} '
                       f'cka_f={met["cka_final"]:.3f}') if 'cka_early' in met else f'cka={met["cka"]:.4f}'
            logger.info(f'│ [{ci+1:2d}/{pop_sz}] '
                        f'fit={fit:+.4f}  AUC={met["auc"]:.4f}  '
                        f'bal={met["bal_acc"]:.4f}  F1={met["f1"]:.4f}  '
                        f'rec={met["recall"]:.4f}  prec={met["precision"]:.4f}  '
                        f'{cka_str}  conf={met["confidence"]:.4f}')

        # ── Generation summary ────────────────────────────────────────────
        gen_best_i = int(np.argmax(gen_fits))
        gen_best_f = gen_fits[gen_best_i]
        gen_best_m = gen_mets[gen_best_i]
        mean_fit   = float(np.mean(gen_fits))
        worst_fit  = float(np.min(gen_fits))
        mean_auc   = float(np.mean([m['auc'] for m in gen_mets]))
        diversity  = float(np.std(np.vstack(population)))

        improved = gen_best_f > best_fitness
        if improved:
            best_fitness = gen_best_f
            best_genome  = population[gen_best_i].copy()
            best_val_met = {k: v for k, v in gen_best_m.items()
                            if k not in ('scores', 'labels')}
            bm = make_model(best_genome)
            torch.save(bm.state_dict(), weights_pth)
            del bm
            if str(device) == 'cuda': torch.cuda.empty_cache()
            stagnation = 0
        else:
            stagnation += 1

        history['best_fit'].append(float(gen_best_f))
        history['mean_fit'].append(mean_fit)
        history['worst_fit'].append(worst_fit)
        history['best_auc'].append(float(gen_best_m['auc']))
        history['best_bal'].append(float(gen_best_m['bal_acc']))
        history['best_f1'].append(float(gen_best_m['f1']))
        history['mean_auc'].append(mean_auc)
        history['diversity'].append(diversity)
        history['best_prec'].append(float(gen_best_m['precision']))
        history['best_rec'].append(float(gen_best_m['recall']))
        history['best_cka'].append(float(gen_best_m['cka']))
        history['best_conf'].append(float(gen_best_m['confidence']))

        star = ' ⭐ NEW BEST' if improved else ''
        logger.info(f'├─ Gen summary  best={gen_best_f:.4f}  mean={mean_fit:.4f}  '
                    f'worst={worst_fit:.4f}  div={diversity:.4f}  ({time.time()-t0:.0f}s){star}')
        logger.info(f'│  Gen best:  AUC={gen_best_m["auc"]:.4f}  '
                    f'bal={gen_best_m["bal_acc"]:.4f}  F1={gen_best_m["f1"]:.4f}  '
                    f'conf={gen_best_m["confidence"]:.4f}')
        logger.info(f'│  Global:    fit={best_fitness:.4f}  stagnation={stagnation} gen(s)')
        if best_genome is not None:
            logger.info(f'│  Genome:    {[f"{v:.3f}" for v in best_genome]}')

        # ── Diversity injection ───────────────────────────────────────────
        if (config.get('diversity_injection') and
                stagnation >= config.get('diversity_patience', 5)):
            n_inj   = max(1, pop_sz // 4)
            worst_i = np.argsort(gen_fits)[:n_inj]
            for wi in worst_i:
                population[wi] = rnd_genome()
            logger.info(f'│  💉 Diversity injection: replaced {n_inj} worst (stagnation reset)')
            stagnation = 0

        # ── Selection + reproduction ──────────────────────────────────────
        new_pop = [best_genome.copy()]   # elitism: carry best forward
        while len(new_pop) < pop_sz:
            pa    = tournament(population, gen_fits)
            pb    = tournament(population, gen_fits)
            child = crossover(pa, pb)
            if np.random.rand() < 0.20:
                child = mutate(child, gen)
            new_pop.append(child)
        population = new_pop

        # ── Save mid-run checkpoint ───────────────────────────────────────
        ec_out = {
            'last_gen':    gen,
            'population':  [g.tolist() for g in population],
            'best_genome': best_genome.tolist(),
            'best_fitness': float(best_fitness),
            'best_val_met': best_val_met,
            'stagnation':  stagnation,
            'history':     history,
        }
        with open(evo_ckpt, 'w') as f:
            json.dump(ec_out, f)
        logger.info(f'└─ Gen {gen+1} complete. Checkpoint saved.')

    logger.info(f'')
    logger.info(f'🏆 [{config["name"]}] Evolution finished!')
    logger.info(f'   Best fitness : {best_fitness:.4f}')
    logger.info(f'   Best val AUC : {best_val_met["auc"]:.4f}')
    logger.info(f'   Best val bal : {best_val_met["bal_acc"]:.4f}')
    logger.info(f'   Best val F1  : {best_val_met["f1"]:.4f}')

    return best_val_met, history, best_genome


logger.info('MeGA-IA engine defined.')

## Experiment Configurations

In [ ]:
# ── Experiment Configurations ──────────────────────────────────────────────────
# v2 experiments start from E01.
# E01 = MeGA-IA v2 Full (AUC + raised penalties + 3-seg alpha + multilayer CKA
#        + confidence bonus + blend crossover) — the deduced best config from v1.
#
# To add further experiments: append dicts following the schema below.

EXPERIMENTS = [

    # ── v2 Experiments ────────────────────────────────────────────────────
    dict(id='E01_mega_ia_v2',   name='E01 MeGA-IA v2 (Full)',
         group='v2 — Integrated',
         fitness_type='auc',      lambda1=0.30, lambda2=0.15,
         penalty_mode='confidence', crossover_mode='blend',
         generations=10, pop_size=15, alpha_mode='segment3', cka_mode='multilayer',
         diversity_injection=True,  diversity_patience=5, sigma_base=0.10,
         description='AUC - λ1·CKA_multi + λ2·Confidence | 3-seg α | blend crossover | deduced best from v1'),

    # ── Placeholder for future experiments ────────────────────────────────
    # dict(id='E02_...', name='E02 ...', group='v2 — ...', ...),
    # dict(id='E03_...', name='E03 ...', group='v2 — ...', ...),
]

# ── Overview table ────────────────────────────────────────────────────────────
print(f'{"ID":<22} {"Group":<25} {"Fit":<9} {"λ1":<5} {"λ2":<5} '
      f'{"G":<4} {"P":<4} {"Alpha":<10} {"CKA":<12} {"Pen":<12} {"Status"}')
print('─' * 120)
for e in EXPERIMENTS:
    status = '✅ done' if load_exp_results(str(CKPT_DIR), e['id']) else ''
    print(f'{e["id"]:<22} {e["group"]:<25} {e["fitness_type"]:<9} '
          f'{e["lambda1"]:<5} {e["lambda2"]:<5} {e["generations"]:<4} '
          f'{e["pop_size"]:<4} {e["alpha_mode"]:<10} {e["cka_mode"]:<12} '
          f'{e.get("penalty_mode","entropy"):<12} {status}')

## Pre-extract Parent 1 Reference Features

In [ ]:
# ── Pre-Extract Parent 1 Reference Features ────────────────────────────────────
# Runs once on the full val_loader. Reused by all experiments.

logger.info('Extracting Parent 1 reference features on full val set...')
logger.info('(This runs once — reused by all experiments)')

p1_model = RawNet(d_args, DEVICE).to(DEVICE)
p1_model.load_state_dict(p1_state)
p1_model.eval()

p1_feats = extract_reference_features(p1_model, val_loader, DEVICE)
del p1_model
if str(DEVICE) == 'cuda': torch.cuda.empty_cache()

for key, tensor in p1_feats.items():
    logger.info(f'  {key:6s}: {tuple(tensor.shape)}  dtype={tensor.dtype}')

logger.info('Parent 1 features ready.')

## Run All Experiments

In [ ]:
# ── Run All Experiments ────────────────────────────────────────────────────────
# Each experiment checkpoints every generation.
# Interrupt freely — resumes from last completed generation on rerun.

completed_results = {}   # exp_id → results dict

for cfg in EXPERIMENTS:
    eid = cfg['id']
    sep = '═' * 70

    existing = load_exp_results(str(CKPT_DIR), eid)
    if existing:
        completed_results[eid] = existing
        logger.info(f'{sep}')
        logger.info(f'[{cfg["name"]}] Already completed — loaded from disk.')
        logger.info(f'  Best val AUC={existing["best_val_metrics"]["auc"]:.4f}  '
                    f'bal={existing["best_val_metrics"]["bal_acc"]:.4f}  '
                    f'F1={existing["best_val_metrics"]["f1"]:.4f}')
        continue

    logger.info(f'{sep}')
    logger.info(f'Starting [{cfg["name"]}]')
    logger.info(f'  {cfg["description"]}')
    logger.info(f'  fitness={cfg["fitness_type"]}  λ1={cfg["lambda1"]}  λ2={cfg["lambda2"]}  '
                f'penalty={cfg.get("penalty_mode","entropy")}')
    logger.info(f'  gens={cfg["generations"]}  pop={cfg["pop_size"]}  '
                f'alpha={cfg["alpha_mode"]}  cka={cfg["cka_mode"]}  '
                f'crossover={cfg.get("crossover_mode","uniform")}')
    logger.info(f'  diversity_injection={cfg["diversity_injection"]}')

    exp_start = time.time()

    best_val_met, history, best_genome = run_mega_ia(
        cfg, p1_state, p2_state, d_args, DEVICE,
        val_loader, p1_feats, str(CKPT_DIR)
    )

    elapsed = time.time() - exp_start
    logger.info(f'[{cfg["name"]}] Finished in {elapsed/60:.1f} min')

    save_exp_results(str(CKPT_DIR), eid, best_val_met, history, best_genome)
    completed_results[eid] = load_exp_results(str(CKPT_DIR), eid)
    logger.info(f'Results saved → checkpoints/{eid}_results.json')

logger.info('')
logger.info(f'All experiments: {len(completed_results)}/{len(EXPERIMENTS)} completed.')

## Evaluate Best Models on Test Set (Deepfake-Eval-2024)
##                      on Test Set (LA)
##                      on Test Set (In the Wild)

In [ ]:
# ── Evaluate Best Models on All Test Sets ─────────────────────────────────────
# For each experiment: loads best_weights.pth and evaluates on:
#   1. dfe_full  — 100% Deepfake-Evals-2024 (primary benchmark)
#   2. la        — LA test set
#   3. itw       — In-the-Wild test set

TEST_LOADERS = {
    'dfe_full': dfe_full_loader,
    'la':       la_loader,
    'itw':      itw_loader,
}

all_test_results = {}   # exp_id → {dataset_key → metrics}

for cfg in EXPERIMENTS:
    eid     = cfg['id']
    wpath   = os.path.join(str(CKPT_DIR), f'{eid}_best_weights.pth')
    all_test_results[eid] = {}

    if not os.path.exists(wpath):
        logger.warning(f'[{cfg["name"]}] No weights file — skipping test eval.')
        continue

    # Load model once, evaluate on all three test sets
    model = RawNet(d_args, DEVICE).to(DEVICE)
    model.load_state_dict(torch.load(wpath, map_location=DEVICE, weights_only=True))
    model.eval()

    for ds_key, loader in TEST_LOADERS.items():
        existing = load_test_results(str(CKPT_DIR), eid, ds_key)
        if existing:
            all_test_results[eid][ds_key] = existing
            logger.info(f'[{cfg["name"]}] [{ds_key}] Already evaluated — '
                        f'AUC={existing["auc"]:.4f}')
            continue

        logger.info(f'[{cfg["name"]}] Evaluating on {ds_key}...')
        all_scores, all_labels = [], []
        with torch.no_grad():
            for x, y in tqdm(loader, desc=f'  {eid} / {ds_key}', leave=False):
                out = model(x.to(DEVICE), is_test=True)
                all_scores.append(out[:, 1].cpu().numpy())
                all_labels.append(y.numpy())

        scores = np.concatenate(all_scores)
        labels = np.concatenate(all_labels)
        preds  = (scores > 0.5).astype(int)

        tm = full_metrics(labels, preds, scores)
        tm['scores'] = scores.tolist()
        tm['labels'] = labels.tolist()

        all_test_results[eid][ds_key] = tm
        save_test_results(str(CKPT_DIR), eid, ds_key, tm)
        logger.info(f'  [{ds_key}] AUC={tm["auc"]:.4f}  bal={tm["bal_acc"]:.4f}  '
                    f'F1={tm["f1"]:.4f}  acc={tm["accuracy"]:.4f}')

    del model
    if str(DEVICE) == 'cuda': torch.cuda.empty_cache()

# ── Summary table ─────────────────────────────────────────────────────────────
logger.info('')
logger.info(f'{"Experiment":<22} {"DFE AUC":>8} {"DFE F1":>7} {"LA AUC":>7} {"LA F1":>6} {"ITW AUC":>8} {"ITW F1":>7}')
logger.info('─' * 75)
for cfg in EXPERIMENTS:
    eid = cfg['id']
    r   = all_test_results.get(eid, {})
    def _g(ds, k): return f'{r[ds][k]:.4f}' if ds in r and k in r[ds] else '  —   '
    logger.info(f'{eid:<22} {_g("dfe_full","auc"):>8} {_g("dfe_full","f1"):>7} '
                f'{_g("la","auc"):>7} {_g("la","f1"):>6} '
                f'{_g("itw","auc"):>8} {_g("itw","f1"):>7}')

logger.info(f'Test evaluation complete: {sum(bool(v) for v in all_test_results.values())}/{len(EXPERIMENTS)} models.')

## Visualization

In [ ]:
# ── Plot style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f8f8',
    'axes.grid': True, 'grid.color': '#e0e0e0', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 8, 'legend.framealpha': 0.9,
})

# Assign each experiment a distinct color
PALETTE = plt.cm.tab10.colors
EXP_COLOR = {cfg['id']: PALETTE[i % 10] for i, cfg in enumerate(EXPERIMENTS)}
GROUP_STYLE = {
    'Tier 1 — Fitness':          {'ls': '-',  'marker': 'o'},
    'Tier 2 — Search Budget':    {'ls': '--', 'marker': 's'},
    'Tier 3 — Penalty Ablation': {'ls': ':',  'marker': '^'},
}

def group_style(grp):
    return GROUP_STYLE.get(grp, {'ls': '-', 'marker': 'o'})


def fig_fitness_curves():
    """Plot best + mean fitness and best AUC per generation for every experiment."""
    n = len(completed_results)
    if n == 0: return
    ncols = 2
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5))
    axes = np.array(axes).flatten()

    for ax_i, cfg in enumerate(EXPERIMENTS):
        eid = cfg['id']
        if eid not in completed_results: continue
        h   = completed_results[eid]['history']
        ax  = axes[ax_i]
        gens = list(range(1, len(h['best_fit']) + 1))
        clr = EXP_COLOR[eid]

        ax.plot(gens, h['best_fit'],  color=clr, lw=2,   label='Best fitness')
        ax.plot(gens, h['mean_fit'],  color=clr, lw=1.2, ls='--', alpha=0.7, label='Mean fitness')
        ax.plot(gens, h['best_auc'],  color='gray', lw=1.5, ls=':',  label='Best AUC (val)')
        ax.set_title(f'{cfg["name"]}', fontweight='bold')
        ax.set_xlabel('Generation')
        ax.set_ylabel('Score')
        ax.legend(loc='lower right')

    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle('Fitness Evolution per Experiment', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig1_fitness_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 1 saved: fitness curves.', 'OK')


def fig_val_metrics_evolution():
    """Val balanced-acc, F1, recall per generation for every experiment."""
    n = len(completed_results)
    if n == 0: return
    ncols = 2
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5))
    axes = np.array(axes).flatten()

    for ax_i, cfg in enumerate(EXPERIMENTS):
        eid = cfg['id']
        if eid not in completed_results: continue
        h   = completed_results[eid]['history']
        ax  = axes[ax_i]
        gens = list(range(1, len(h['best_fit']) + 1))
        clr = EXP_COLOR[eid]

        ax.plot(gens, h['best_bal'],  color=clr,      lw=2,   label='Bal. acc')
        ax.plot(gens, h['best_f1'],   color='steelblue', lw=1.5, ls='--', label='F1')
        ax.plot(gens, h['best_rec'],  color='coral',  lw=1.2, ls=':',  label='Recall')
        ax.plot(gens, h['best_prec'], color='seagreen', lw=1.2, ls='-.', label='Precision')
        ax.set_title(cfg['name'], fontweight='bold')
        ax.set_xlabel('Generation')
        ax.set_ylabel('Metric')
        ax.set_ylim(0, 1.05)
        ax.legend(loc='best')

    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle('Validation Metric Evolution per Experiment', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig2_val_metrics.png'), dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 2 saved: val metric curves.', 'OK')


def fig_test_comparison():
    """Grouped bar chart of test metrics for all experiments."""
    if not all_test_results: return
    metrics_shown = ['auc', 'bal_acc', 'accuracy', 'f1', 'precision', 'recall']
    metric_labels = ['AUC', 'Bal Acc', 'Accuracy', 'F1', 'Precision', 'Recall']

    exp_list = [cfg for cfg in EXPERIMENTS if cfg['id'] in all_test_results]
    names    = [cfg['name'] for cfg in exp_list]
    n_exp    = len(exp_list)
    n_met    = len(metrics_shown)

    x = np.arange(n_exp)
    w = 0.13
    offsets = np.linspace(-(n_met-1)/2, (n_met-1)/2, n_met) * w

    fig, ax = plt.subplots(figsize=(max(14, n_exp * 1.4), 5))
    bar_colors = plt.cm.Set2.colors

    for mi, (mkey, mlabel) in enumerate(zip(metrics_shown, metric_labels)):
        vals = [all_test_results[cfg['id']][mkey] for cfg in exp_list]
        bars = ax.bar(x + offsets[mi], vals, w * 0.9, label=mlabel,
                      color=bar_colors[mi % len(bar_colors)], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha='right')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color='red', ls='--', lw=0.8, alpha=0.5, label='Random baseline')
    ax.legend(loc='upper right', ncol=3)
    ax.set_title('Test Set Metrics — All Experiments (Deepfake-Eval-2024)', fontweight='bold')

    # Shade by group
    group_colors = {'Tier 1 — Fitness': '#fff3e0',
                    'Tier 2 — Search Budget': '#e8f5e9',
                    'Tier 3 — Penalty Ablation': '#e3f2fd'}
    prev_grp, grp_start = None, 0
    for i, cfg in enumerate(exp_list):
        grp = cfg['group']
        if grp != prev_grp:
            if prev_grp is not None:
                ax.axvspan(grp_start - 0.5, i - 0.5,
                           color=group_colors.get(prev_grp, 'white'), alpha=0.25, zorder=0)
            grp_start = i
            prev_grp = grp
    ax.axvspan(grp_start - 0.5, n_exp - 0.5,
               color=group_colors.get(prev_grp, 'white'), alpha=0.25, zorder=0)

    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig3_test_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 3 saved: test comparison bars.', 'OK')


def fig_roc_curves():
    """Overlaid ROC curves on test set."""
    if not all_test_results: return
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Random (AUC=0.50)')

    for cfg in EXPERIMENTS:
        eid = cfg['id']
        if eid not in all_test_results: continue
        tm = all_test_results[eid]
        scores = np.array(tm['scores'])
        labels = np.array(tm['labels'])
        fpr, tpr, _ = roc_curve(labels, scores)
        auc_val = sk_auc(fpr, tpr)
        gs = group_style(cfg['group'])
        ax.plot(fpr, tpr, color=EXP_COLOR[eid],
                lw=1.8, ls=gs['ls'],
                label=f'{cfg["name"]} (AUC={auc_val:.4f})')

    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves — Deepfake-Eval-2024 Test Set', fontweight='bold')
    ax.legend(loc='lower right', fontsize=7.5)
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig4_roc_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 4 saved: ROC curves.', 'OK')


def fig_confusion_matrices():
    """Grid of confusion matrices for all experiments."""
    if not all_test_results: return
    exp_list = [cfg for cfg in EXPERIMENTS if cfg['id'] in all_test_results]
    n = len(exp_list)
    ncols = min(5, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = np.array(axes).flatten()

    for i, cfg in enumerate(exp_list):
        ax  = axes[i]
        tm  = all_test_results[cfg['id']]
        cm  = np.array(tm['confusion_matrix'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Pred Real', 'Pred Fake'],
                    yticklabels=['Act Real',  'Act Fake'],
                    cbar=False, linewidths=0.5)
        ax.set_title(cfg['name'], fontsize=8, fontweight='bold')
        auc_v = tm['auc']
        bal_v = tm['bal_acc']
        ax.set_xlabel(f'AUC={auc_v:.3f}  bal={bal_v:.3f}', fontsize=7.5)

    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle('Confusion Matrices — Deepfake-Eval-2024 Test Set',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig5_confusion_matrices.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 5 saved: confusion matrices.', 'OK')


def fig_diversity_and_cka():
    """Population diversity + CKA penalty across generations."""
    n = len(completed_results)
    if n == 0: return
    ncols = 2
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.2))
    axes = np.array(axes).flatten()

    for ax_i, cfg in enumerate(EXPERIMENTS):
        eid = cfg['id']
        if eid not in completed_results: continue
        h   = completed_results[eid]['history']
        ax  = axes[ax_i]
        gens = list(range(1, len(h['best_fit']) + 1))
        clr = EXP_COLOR[eid]

        ax2 = ax.twinx()
        ax.plot(gens, h['diversity'], color=clr, lw=2, label='Population diversity')
        if 'mean_auc' in h:
            ax2.plot(gens, h['mean_auc'], color='dimgray', lw=1.2, ls='--',
                     label='Mean pop AUC')
        ax.set_title(cfg['name'], fontweight='bold')
        ax.set_xlabel('Generation')
        ax.set_ylabel('Diversity (genome std)', color=clr)
        ax2.set_ylabel('Mean pop AUC', color='dimgray')
        lines1, labs1 = ax.get_legend_handles_labels()
        lines2, labs2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labs1 + labs2, loc='best', fontsize=7.5)

    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle('Population Diversity & Mean AUC per Generation',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(str(CKPT_DIR), 'fig6_diversity.png'), dpi=150, bbox_inches='tight')
    plt.show()
    logger.info('Figure 6 saved: diversity curves.', 'OK')


logger.info('All plotting functions defined.', 'OK')


## Generate All Figures

In [ ]:
logger.info('Generating Figure 1: Fitness evolution curves...', 'HEAD')
fig_fitness_curves()

logger.info('Generating Figure 2: Validation metric evolution...', 'HEAD')
fig_val_metrics_evolution()

logger.info('Generating Figure 3: Test set comparison bars...', 'HEAD')
fig_test_comparison()

logger.info('Generating Figure 4: ROC curves...', 'HEAD')
fig_roc_curves()

logger.info('Generating Figure 5: Confusion matrices...', 'HEAD')
fig_confusion_matrices()

logger.info('Generating Figure 6: Diversity + mean AUC...', 'HEAD')
fig_diversity_and_cka()


## Final Summary

In [ ]:
print()
print('=' * 110)
print(f'  FINAL RESULTS SUMMARY — MeGA-IA Experiments vs Deepfake-Eval-2024 Test Set')
print('=' * 110)
print(f'  {"Experiment":<30} {"Group":<25} {"AUC":>6} {"BalAcc":>7} {"Acc":>6} '
      f'{"F1":>6} {"Prec":>6} {"Rec":>6}  {"Val AUC":>8}')
print('─' * 110)

# Sort by test AUC descending (show best first)
ranked = sorted(
    [cfg for cfg in EXPERIMENTS if cfg['id'] in all_test_results],
    key=lambda c: all_test_results[c['id']]['auc'], reverse=True
)

RANK_ICONS = ['🥇', '🥈', '🥉']
for rank, cfg in enumerate(ranked):
    eid = cfg['id']
    tm  = all_test_results[eid]
    vm  = completed_results.get(eid, {}).get('best_val_metrics', {})
    icon = RANK_ICONS[rank] if rank < 3 else f'  {rank+1}.'
    print(f'{icon} {cfg["name"]:<28} {cfg["group"]:<25} '
          f'{tm["auc"]:>6.4f} {tm["bal_acc"]:>7.4f} {tm["accuracy"]:>6.4f} '
          f'{tm["f1"]:>6.4f} {tm["precision"]:>6.4f} {tm["recall"]:>6.4f}  '
          f'{vm.get("auc", float("nan")):>8.4f}')

print('─' * 110)

# ── Best experiment ──────────────────────────────────────────────────────────
if ranked:
    best = ranked[0]
    bm   = all_test_results[best['id']]
    print()
    print('🏆 BEST EXPERIMENT:', best['name'])
    print(f'   Config : {best["description"]}')
    print(f'   Test   : AUC={bm["auc"]:.4f}  BalAcc={bm["bal_acc"]:.4f}  '
          f'Acc={bm["accuracy"]:.4f}  F1={bm["f1"]:.4f}')
    print(f'   TP={bm["tp"]}  TN={bm["tn"]}  FP={bm["fp"]}  FN={bm["fn"]}')

# ── Tier-level analysis ──────────────────────────────────────────────────────
print()
print('── PER-TIER BEST PERFORMANCE ─────────────────────────────────────────────')
tiers = {}
for cfg in EXPERIMENTS:
    eid = cfg['id']
    if eid not in all_test_results: continue
    grp = cfg['group']
    auc_val = all_test_results[eid]['auc']
    if grp not in tiers or auc_val > tiers[grp]['auc']:
        tiers[grp] = {'name': cfg['name'], 'auc': auc_val,
                      'bal': all_test_results[eid]['bal_acc'],
                      'f1':  all_test_results[eid]['f1']}

for grp, info in tiers.items():
    print(f'  {grp:<30}  best={info["name"]:<28}  '
          f'AUC={info["auc"]:.4f}  bal={info["bal"]:.4f}  F1={info["f1"]:.4f}')

# ── Biggest improvement from each change ───────────────────────────────────
print()
print('── IMPACT ANALYSIS ──────────────────────────────────────────────────────')
base_auc = all_test_results.get('E01_original', {}).get('auc', float('nan'))
print(f'  Baseline (E01 original MeGA-IA) test AUC: {base_auc:.4f}')
print()
for cfg in EXPERIMENTS[1:]:
    eid = cfg['id']
    if eid not in all_test_results: continue
    delta = all_test_results[eid]['auc'] - base_auc
    sign  = '+' if delta >= 0 else ''
    bar   = '█' * int(abs(delta) * 200)
    print(f'  {cfg["name"]:<35}  ΔAUC={sign}{delta:.4f}  {bar}')

print()
print('── CHECKPOINTED FILES ────────────────────────────────────────────────────')
for f in sorted(os.listdir(str(CKPT_DIR))):
    fpath = os.path.join(str(CKPT_DIR), f)
    print(f'  {f:<55}  {os.path.getsize(fpath)/1e3:>8.1f} KB')

print()
print('=' * 110)
print('  All figures saved to:', str(CKPT_DIR))
print('=' * 110)


## Using 15% Deepfake-Evals as validation set for fitness function for best experiment model

In [ ]:
#split Deepfake-Evals into 2 parts 15% Validation and 85% Test. Be careful don't overlap them. Run the best experiment again but this time, use 15% Deepfake-Evals as validation set during fitness function and test on 85% Deepfake-Evals. 85% Deepfake evals shouldn't contain any file from previoius 15%